<a href="https://colab.research.google.com/github/sergiocostaifes/PPCOMP_DM/blob/main/notebooks/03_window_5min_base.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NB03 — Construção de Janelas Temporais (5 minutos)

## 1. Contexto

Após a limpeza e normalização temporal realizada no NB02, este notebook transforma os
eventos individuais em uma representação agregada no tempo, adequada para análise
estatística e modelagem subsequente.

A modelagem baseada apenas em eventos isolados apresenta limitações para o estudo do
comportamento operacional ao longo do tempo. Por isso, torna-se necessário discretizar a
linha temporal em janelas fixas e construir uma série agregada, preservando continuidade,
rastreabilidade e capacidade de interpretação.

## 2. Objetivo

O objetivo deste notebook é:
- discretizar o tempo em janelas fixas de 5 minutos;
- agregar eventos dentro de cada janela;
- construir métricas representativas do comportamento do sistema;
- produzir uma série temporal contínua, incluindo janelas sem eventos;
- documentar explicitamente a estrutura temporal resultante da agregação.

## 3. Papel no Pipeline Atualizado

Este notebook representa a passagem do nível de eventos individuais para o nível de
comportamento agregado do sistema.

Seu papel é estabelecer a unidade temporal de análise que será utilizada nas etapas
seguintes, permitindo:
- observar o comportamento do sistema em escala regular;
- consolidar métricas operacionais por janela;
- construir uma série contínua adequada à detecção de episódios e à engenharia de atributos.

Assim, o NB03 fornece a base temporal agregada sobre a qual serão realizadas as análises
posteriores do pipeline.

## 4. Principais Transformações

As transformações realizadas incluem:
- definição de janelas fixas de 5 minutos;
- mapeamento de eventos para identificadores de janela (`bucket_id`);
- agregação de métricas por janela;
- construção de uma série temporal contínua, incluindo janelas sem eventos;
- geração de diagnósticos sobre a cobertura temporal e o bucket inicial da série.

## 5. Estrutura das Métricas

As métricas agregadas por janela incluem:

### 5.1 Volume
- número total de eventos;
- número total de falhas;
- número de máquinas distintas;
- número de coleções distintas.

### 5.2 Intensidade / contexto operacional
- média de prioridade;
- média de CPU solicitada;
- média de memória solicitada.

### 5.3 Disponibilidade das requisições
- proporção de registros com CPU solicitada disponível;
- proporção de registros com memória solicitada disponível.

### 5.4 Tipologia de eventos
- contagens por tipo de evento relevante, como `FAIL`, `SCHEDULE`, `FINISH`,
  `ENABLE`, `LOST`, `EVICT` e `KILL`.

## 6. Saídas Esperadas

Ao final deste notebook, espera-se obter:
- dataset agregado por janelas de 5 minutos;
- série temporal contínua, incluindo janelas vazias;
- artefatos persistidos para uso nas próximas etapas;
- resumo estruturado da agregação temporal realizada.

## 7. Relação com as Próximas Etapas

Este notebook alimenta diretamente:
- NB04 — Detecção de episódios críticos;
- NB05 — Engenharia de atributos;
- etapas posteriores do pipeline que dependem de uma série temporal regular e auditável.

A qualidade da discretização e da agregação impacta diretamente a qualidade dos episódios,
das features e dos artefatos analíticos derivados.

## 8. Observações Metodológicas

A escolha da janela de 5 minutos representa um compromisso entre granularidade temporal
e estabilidade analítica.

A inclusão de janelas sem eventos é importante para preservar continuidade temporal e
evitar que lacunas artificiais prejudiquem a interpretação da série.

Como a execução atual do NB02 preservou a `hour == 0`, espera-se que esta etapa reflita
essa decisão na estrutura agregada, idealmente com presença do bucket inicial da série.

Este notebook estabelece a base formal da série temporal utilizada no restante do pipeline.

In [1]:
# ============================================================
# 03_window_5min_base.ipynb
# Construção da série temporal agregada em janelas de 5 minutos
# ============================================================

# =========================
# 0) Bootstrap seguro
# =========================
from pathlib import Path
import os
import sys
import subprocess
import importlib
import json
import ast

# Mount seguro
if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("[Bootstrap] Drive já montado.")

REPO_DIR = Path("/content/drive/MyDrive/Mestrado/PPCOMP_DM")
GITHUB_REPO = "https://github.com/sergiocostaifes/PPCOMP_DM.git"

if not REPO_DIR.exists():
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", GITHUB_REPO, str(REPO_DIR)], check=True)

repo_str = str(REPO_DIR)
if repo_str not in sys.path:
    sys.path.insert(0, repo_str)

importlib.invalidate_caches()

from src.paths import PROCESSED_PATH, FEATURES_PATH, REPORTS_PATH, ensure_dirs
ensure_dirs()

def log(msg: str) -> None:
    print(f"[03_window_5min_base] {msg}")


# =========================
# 1) Parâmetros
# =========================
import pandas as pd
import numpy as np
from IPython.display import display

BUCKET_SEC = 5 * 60
BUCKET_US = BUCKET_SEC * 1_000_000

TOP_EVENTS = ["FAIL", "SCHEDULE", "FINISH", "ENABLE", "LOST", "EVICT", "KILL"]


# =========================
# 2) Metadados do NB02
# =========================
NB02_SUMMARY_FILE = REPORTS_PATH / "02_clean_normalize_summary.json"



nb02_summary = {}
if NB02_SUMMARY_FILE.exists():
    nb02_summary = json.loads(NB02_SUMMARY_FILE.read_text(encoding="utf-8"))
    log(f"Resumo do NB02 carregado: {NB02_SUMMARY_FILE}")
else:
    log("Resumo do NB02 não encontrado; seguindo sem metadados adicionais.")

SCENARIO_LABEL = nb02_summary.get("scenario_label", "unknown")
REMOVE_HOUR_ZERO = nb02_summary.get("remove_hour_zero", None)

# =========================
# 3) Leitura do dataset limpo
# =========================
CLEAN_PARQUET = PROCESSED_PATH / "google_trace_clean.parquet"
assert CLEAN_PARQUET.exists(), f"Arquivo não encontrado: {CLEAN_PARQUET}"

df = pd.read_parquet(CLEAN_PARQUET)

log(f"Shape entrada: {df.shape}")

required_cols = [
    "t_rel_us",
    "machine_id",
    "collection_id",
    "event",
    "failed",
    "priority",
]
missing = [c for c in required_cols if c not in df.columns]
assert not missing, f"Colunas ausentes: {missing}"

df["t_rel_us"] = pd.to_numeric(df["t_rel_us"], errors="coerce")
df = df.dropna(subset=["t_rel_us"]).copy()
df["t_rel_us"] = df["t_rel_us"].astype("int64")

assert (df["t_rel_us"] >= 0).all(), "Foram encontrados valores negativos em t_rel_us."

df = df.sort_values(["t_rel_us", "time"] if "time" in df.columns else ["t_rel_us"]).reset_index(drop=True)

t_rel_min = int(df["t_rel_us"].min())
t_rel_max = int(df["t_rel_us"].max())

hour_min_input = int(df["hour"].min()) if "hour" in df.columns else None
hour_max_input = int(df["hour"].max()) if "hour" in df.columns else None

log(f"t_rel_us range: {t_rel_min}..{t_rel_max}")
if hour_min_input is not None and hour_max_input is not None:
    log(f"hour range entrada: {hour_min_input}..{hour_max_input}")

# =========================
# 4) Definição dos buckets
# =========================
df["bucket_id"] = (df["t_rel_us"] // BUCKET_US).astype("int64")
df["bucket_start_us"] = df["bucket_id"] * BUCKET_US

bucket_min_raw = int(df["bucket_id"].min())
bucket_max_raw = int(df["bucket_id"].max())
bucket_nunique_raw = int(df["bucket_id"].nunique())

log(f"bucket range bruto: {bucket_min_raw}..{bucket_max_raw}")
log(f"Buckets distintos brutos: {bucket_nunique_raw}")

if t_rel_min == 0:
    assert bucket_min_raw == 0, (
        "Com t_rel_us iniciando em 0, espera-se bucket_id inicial igual a 0."
    )
    log("Checkpoint OK: bucket inicial consistente com preservação do início da série.")

# =========================
# 5) Parse de resource_request
# =========================


def parse_dict(x):
    """
    Converte resource_request para dict quando possível.
    Suporta:
    - dict já pronto
    - string JSON (aspas duplas)
    - string estilo Python dict (aspas simples) via ast.literal_eval
    Retorna None se não conseguir parsear ou se não for dict.
    """
    if isinstance(x, dict):
        return x

    if isinstance(x, str):
        s = x.strip()
        if not s:
            return None

        # JSON padrão
        try:
            v = json.loads(s)
            return v if isinstance(v, dict) else None
        except Exception:
            pass

        # Fallback para formato dict Python
        try:
            v = ast.literal_eval(s)
            return v if isinstance(v, dict) else None
        except Exception:
            return None

    return None


if "resource_request" in df.columns:
    rr = df["resource_request"].map(parse_dict)
    df["req_cpus"] = rr.map(lambda d: d.get("cpus") if isinstance(d, dict) else np.nan)
    df["req_mem"] = rr.map(lambda d: d.get("memory") if isinstance(d, dict) else np.nan)
else:
    df["req_cpus"] = np.nan
    df["req_mem"] = np.nan


valid_cpu = int(df["req_cpus"].notna().sum())
valid_mem = int(df["req_mem"].notna().sum())

log(f"req_cpus válidos: {valid_cpu} ({valid_cpu / len(df):.4%})")
log(f"req_mem válidos: {valid_mem} ({valid_mem / len(df):.4%})")


df["has_req_cpus"] = df["req_cpus"].notna().astype("int64")
df["has_req_mem"] = df["req_mem"].notna().astype("int64")


# =========================
# 6) Agregações por bucket
# =========================
base = (
    df.groupby("bucket_id", as_index=False)
      .agg(
          bucket_start_us=("bucket_start_us", "min"),
          n_events=("event", "size"),
          n_failed=("failed", "sum"),
          n_machines=("machine_id", "nunique"),
          n_collections=("collection_id", "nunique"),
          mean_priority=("priority", "mean"),
          mean_req_cpus=("req_cpus", "mean"),
          mean_req_mem=("req_mem", "mean"),

          req_cpus_presence_rate=("has_req_cpus", "mean"),
          req_mem_presence_rate=("has_req_mem", "mean"),
      )
)

evt_counts = (
    df[df["event"].isin(TOP_EVENTS)]
      .groupby(["bucket_id", "event"], observed=False)
      .size()
      .unstack(fill_value=0)
)

for ev in TOP_EVENTS:
    if ev not in evt_counts.columns:
        evt_counts[ev] = 0

evt_counts = evt_counts[TOP_EVENTS].reset_index()
evt_counts = evt_counts.rename(columns={ev: f"event_{ev}_count" for ev in TOP_EVENTS})

base = base.merge(evt_counts, on="bucket_id", how="left")

count_cols = (
    ["n_events", "n_failed", "n_machines", "n_collections"] +
    [f"event_{ev}_count" for ev in TOP_EVENTS]
)

for c in count_cols:
    base[c] = base[c].fillna(0).astype("int64")

presence_rate_cols = ["req_cpus_presence_rate", "req_mem_presence_rate"]
for c in presence_rate_cols:
    base[c] = base[c].fillna(0.0).astype("float32")

base = base.sort_values("bucket_id").reset_index(drop=True)

base_rows = int(len(base))
base_bucket_min = int(base["bucket_id"].min())
base_bucket_max = int(base["bucket_id"].max())

log(f"Base agregada shape: {base.shape}")
log(f"Base bucket range: {base_bucket_min}..{base_bucket_max}")

# =========================
# 7) Série contínua
# =========================
bmin = base_bucket_min
bmax = base_bucket_max

full = pd.DataFrame({"bucket_id": np.arange(bmin, bmax + 1, dtype=np.int64)})
full["bucket_start_us"] = full["bucket_id"] * BUCKET_US

series = full.merge(base, on=["bucket_id", "bucket_start_us"], how="left")

for c in count_cols:
    series[c] = series[c].fillna(0).astype("int64")

for c in presence_rate_cols:
    series[c] = series[c].fillna(0.0).astype("float32")

series = series.sort_values("bucket_id").reset_index(drop=True)

series_rows = int(len(series))
gap_buckets = int(series_rows - base_rows)
empty_window_ratio = float(gap_buckets / series_rows) if series_rows > 0 else 0.0

log(f"Série contínua shape: {series.shape}")
log(f"Gap buckets: {gap_buckets} ({empty_window_ratio:.4%})")

# =========================
# 8) Diagnósticos adicionais
# =========================
bucket0_present = bool((base["bucket_id"] == 0).any())

if bucket0_present:
    bucket0_row = base.loc[base["bucket_id"] == 0].iloc[0]
    bucket0_summary = {
        "bucket_id": int(bucket0_row["bucket_id"]),
        "bucket_start_us": int(bucket0_row["bucket_start_us"]),
        "n_events": int(bucket0_row["n_events"]),
        "n_failed": int(bucket0_row["n_failed"]),
        "n_machines": int(bucket0_row["n_machines"]),
        "n_collections": int(bucket0_row["n_collections"]),
    }
else:
    bucket0_summary = None

na_summary_series = {
    c: int(series[c].isna().sum())
    for c in ["mean_priority", "mean_req_cpus", "mean_req_mem"]
    if c in series.columns
}

log(f"Bucket 0 presente na base agregada: {bucket0_present}")
if bucket0_summary is not None:
    log(f"Resumo bucket 0: {bucket0_summary}")

# =========================
# 9) Persistência
# =========================
BASE_FILE = FEATURES_PATH / "window_5min_base.parquet"
SERIES_FILE = FEATURES_PATH / "window_5min_series.parquet"

BASE_SCENARIO_FILE = FEATURES_PATH / f"window_5min_base_{SCENARIO_LABEL}.parquet"
SERIES_SCENARIO_FILE = FEATURES_PATH / f"window_5min_series_{SCENARIO_LABEL}.parquet"

base.to_parquet(BASE_FILE, compression="snappy", index=False)
series.to_parquet(SERIES_FILE, compression="snappy", index=False)

base.to_parquet(BASE_SCENARIO_FILE, compression="snappy", index=False)
series.to_parquet(SERIES_SCENARIO_FILE, compression="snappy", index=False)

log(f"Base salva (canônica): {BASE_FILE}")
log(f"Série salva (canônica): {SERIES_FILE}")
log(f"Base salva (cenário): {BASE_SCENARIO_FILE}")
log(f"Série salva (cenário): {SERIES_SCENARIO_FILE}")

# =========================
# 10) Summary JSON
# =========================
summary = {
    "input_file": str(CLEAN_PARQUET),
    "scenario_label": SCENARIO_LABEL,
    "remove_hour_zero": REMOVE_HOUR_ZERO,

    "rows_in": int(len(df)),
    "t_rel_min": int(t_rel_min),
    "t_rel_max": int(t_rel_max),
    "hour_min_input": hour_min_input,
    "hour_max_input": hour_max_input,

    "bucket_id_min_raw": int(bucket_min_raw),
    "bucket_id_max_raw": int(bucket_max_raw),
    "bucket_id_nunique_raw": int(bucket_nunique_raw),

    "base_rows": int(base_rows),
    "series_rows": int(series_rows),
    "bucket_id_min": int(bmin),
    "bucket_id_max": int(bmax),
    "bucket_span": int(bmax - bmin + 1),
    "gap_buckets": int(gap_buckets),
    "empty_window_ratio": float(empty_window_ratio),

    "bucket0_present": bool(bucket0_present),
    "bucket0_summary": bucket0_summary,

    "avg_events_per_bucket": float(base["n_events"].mean()),
    "avg_failed_per_bucket": float(base["n_failed"].mean()),

    "req_cpu_valid_ratio": float(valid_cpu / len(df)),
    "req_mem_valid_ratio": float(valid_mem / len(df)),
    "top_events": TOP_EVENTS,

    "series_na_summary": na_summary_series,

    "output_base_file_canonical": str(BASE_FILE),
    "output_series_file_canonical": str(SERIES_FILE),
    "output_base_file_scenario": str(BASE_SCENARIO_FILE),
    "output_series_file_scenario": str(SERIES_SCENARIO_FILE),
}

SUMMARY_FILE = REPORTS_PATH / "03_window_5min_base_summary.json"
SUMMARY_SCENARIO_FILE = REPORTS_PATH / f"03_window_5min_base_summary_{SCENARIO_LABEL}.json"

SUMMARY_FILE.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
SUMMARY_SCENARIO_FILE.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")

log(f"Resumo salvo (canônico): {SUMMARY_FILE}")
log(f"Resumo salvo (cenário): {SUMMARY_SCENARIO_FILE}")

# =========================
# 11) Visualização rápida
# =========================
print("\n=== RESUMO RÁPIDO ===")
print(f"Scenario label            : {SCENARIO_LABEL}")
print(f"Remove hour 0             : {REMOVE_HOUR_ZERO}")
print(f"Rows in                   : {len(df)}")
print(f"Bucket range              : {bmin}..{bmax}")
print(f"Base rows                 : {base_rows}")
print(f"Series rows               : {series_rows}")
print(f"Gap buckets               : {gap_buckets}")
print(f"Bucket 0 present          : {bucket0_present}")
print(f"Avg events per bucket     : {base['n_events'].mean():.4f}")
print(f"Avg failed per bucket     : {base['n_failed'].mean():.4f}")

print("\n=== HEAD BASE ===")
display(base.head())

print("\n=== HEAD SERIES ===")
display(series.head())

Mounted at /content/drive
[03_window_5min_base] Resumo do NB02 carregado: /content/drive/MyDrive/Mestrado/04-reports/02_clean_normalize_summary.json
[03_window_5min_base] Shape entrada: (405891, 23)
[03_window_5min_base] t_rel_us range: 0..2678923967375
[03_window_5min_base] hour range entrada: 0..744
[03_window_5min_base] bucket range bruto: 0..8929
[03_window_5min_base] Buckets distintos brutos: 8928
[03_window_5min_base] Checkpoint OK: bucket inicial consistente com preservação do início da série.
[03_window_5min_base] req_cpus válidos: 405117 (99.8093%)
[03_window_5min_base] req_mem válidos: 405117 (99.8093%)
[03_window_5min_base] Base agregada shape: (8928, 18)
[03_window_5min_base] Base bucket range: 0..8929
[03_window_5min_base] Série contínua shape: (8930, 18)
[03_window_5min_base] Gap buckets: 2 (0.0224%)
[03_window_5min_base] Bucket 0 presente na base agregada: True
[03_window_5min_base] Resumo bucket 0: {'bucket_id': 0, 'bucket_start_us': 0, 'n_events': 56452, 'n_failed': 37

,bucket_id,bucket_start_us,n_events,n_failed,n_machines,n_collections,mean_priority,mean_req_cpus,mean_req_mem,req_cpus_presence_rate,req_mem_presence_rate,event_FAIL_count,event_SCHEDULE_count,event_FINISH_count,event_ENABLE_count,event_LOST_count,event_EVICT_count,event_KILL_count
0,0,0,56452,37082,43394,1232,166.682739,0.018434,0.009607,0.986289,0.986289,37082,154,600,18616,0,0,0
1,2,600000000,28,9,28,19,208.464286,0.008100,0.016303,1.000000,1.000000,9,0,7,7,5,0,0
2,3,900000000,32,2,32,18,135.656250,0.009495,0.003928,1.000000,1.000000,2,0,19,4,6,0,1
3,4,1200000000,29,6,28,20,233.068966,0.012932,0.003717,1.000000,1.000000,6,0,6,6,11,0,0
4,5,1500000000,25,5,25,19,253.360000,0.011083,0.012531,1.000000,1.000000,5,0,4,8,8,0,0



=== HEAD SERIES ===


,bucket_id,bucket_start_us,n_events,n_failed,n_machines,n_collections,mean_priority,mean_req_cpus,mean_req_mem,req_cpus_presence_rate,req_mem_presence_rate,event_FAIL_count,event_SCHEDULE_count,event_FINISH_count,event_ENABLE_count,event_LOST_count,event_EVICT_count,event_KILL_count
0,0,0,56452,37082,43394,1232,166.682739,0.018434,0.009607,0.986289,0.986289,37082,154,600,18616,0,0,0
1,1,300000000,0,0,0,0,NaN,NaN,NaN,0.000000,0.000000,0,0,0,0,0,0,0
2,2,600000000,28,9,28,19,208.464286,0.008100,0.016303,1.000000,1.000000,9,0,7,7,5,0,0
3,3,900000000,32,2,32,18,135.656250,0.009495,0.003928,1.000000,1.000000,2,0,19,4,6,0,1
4,4,1200000000,29,6,28,20,233.068966,0.012932,0.003717,1.000000,1.000000,6,0,6,6,11,0,0


## 9. Conclusão da Etapa

A etapa de discretização temporal e construção da série agregada foi concluída com sucesso,
resultando em dois artefatos principais:
- uma base agregada por janelas de 5 minutos (`window_5min_base.parquet`);
- uma série temporal contínua, incluindo janelas sem eventos (`window_5min_series.parquet`).

Nesta execução, a agregação refletiu corretamente a decisão metodológica adotada no NB02,
que preservou a `hour == 0`. Como consequência:
- a série passou a iniciar em `bucket_id = 0`;
- o intervalo agregado cobriu os buckets de 0 a 8929;
- a base agregada resultou em 8.928 janelas observadas;
- a série contínua resultou em 8.930 janelas, com apenas 2 lacunas preenchidas.

Esses resultados indicam que a discretização temporal foi realizada de forma consistente,
preservando a continuidade da linha do tempo e produzindo uma base adequada para as
etapas subsequentes do pipeline.

## 10. Achados Experimentais da Execução Atual

A execução atual produziu diagnósticos importantes sobre a estrutura temporal agregada.

### 10.1 Estrutura geral da agregação

A entrada do notebook continha:
- 405.891 registros;
- `t_rel_us` variando de 0 a 2.678.923.967.375;
- `hour` variando de 0 a 744.

A agregação em janelas de 5 minutos produziu:
- `bucket_id_min_raw = 0`;
- `bucket_id_max_raw = 8929`;
- `bucket_id_nunique_raw = 8928`;
- `base_rows = 8928`;
- `series_rows = 8930`;
- `bucket_span = 8930`.

Esses valores mostram que a série agregada cobre integralmente o intervalo temporal
observado, com discretização regular e alta densidade de ocupação.

### 10.2 Confirmação do bucket inicial

Como a execução do NB02 preservou o início da série temporal, esperava-se a presença
do bucket inicial.

Esse comportamento foi confirmado:
- `bucket_id = 0` está presente na base agregada;
- o checkpoint de consistência foi satisfeito;
- a série contínua também se inicia em `bucket_id = 0`.

Esse resultado é metodologicamente relevante, pois demonstra que a preservação da
primeira hora no NB02 teve reflexo direto e coerente na estrutura agregada do NB03.

### 10.3 Caracterização do bucket 0

O bucket inicial apresentou os seguintes valores:
- `bucket_id = 0`
- `bucket_start_us = 0`
- `n_events = 56452`
- `n_failed = 37082`
- `n_machines = 43394`
- `n_collections = 1232`

Trata-se de um bucket com volume extraordinariamente elevado em relação ao restante
da série, refletindo forte concentração de registros no início do eixo temporal.

Esse comportamento deve ser mantido em observação nas etapas seguintes, não como
evidência automática de erro, mas como característica empírica relevante da base
utilizada nesta execução.

### 10.4 Lacunas temporais e série contínua

A comparação entre base agregada e série contínua indicou:
- 8.928 buckets observados na base;
- 8.930 buckets na série contínua;
- 2 `gap_buckets`;
- `empty_window_ratio = 0,000223964...` (aproximadamente 0,0224%).

Isso mostra que a série temporal apresenta altíssima cobertura, com número muito pequeno
de janelas vazias no intervalo total analisado.

As duas lacunas foram preservadas na série contínua por meio do preenchimento estrutural
dos buckets ausentes, o que é importante para manter regularidade temporal nas etapas
posteriores do pipeline.

### 10.5 Disponibilidade das variáveis de requisição

A qualidade das informações extraídas de `resource_request` foi elevada:
- `req_cpu_valid_ratio = 0,998093...`
- `req_mem_valid_ratio = 0,998093...`

Ou seja, aproximadamente 99,81% dos registros apresentaram valores válidos tanto para
CPU quanto para memória solicitada.

Isso confirma que as variáveis derivadas `mean_req_cpus`, `mean_req_mem`,
`req_cpus_presence_rate` e `req_mem_presence_rate` são adequadas para uso analítico
nas etapas seguintes.

### 10.6 Valores ausentes na série contínua

O resumo da série contínua mostrou:
- 2 valores ausentes em `mean_priority`;
- 2 valores ausentes em `mean_req_cpus`;
- 2 valores ausentes em `mean_req_mem`.

Esses valores ausentes correspondem precisamente às 2 janelas vazias inseridas para
preservação da continuidade temporal da série.

Assim, os `NaN` observados nas métricas médias não indicam falha da agregação, mas
apenas ausência de eventos nas janelas correspondentes.

## 11. Importância para o Pipeline Analítico

O NB03 consolida a mudança de representação do problema:
- de eventos individuais;
- para uma série temporal regular e agregada.

Essa transformação é essencial porque:
- reduz a fragmentação do nível de evento;
- permite análise estatística por unidade temporal fixa;
- fornece a base para detecção de episódios críticos no NB04;
- viabiliza a construção de atributos temporais no NB05.

Além disso, a execução atual fortalece a rastreabilidade metodológica do pipeline ao
mostrar explicitamente que a preservação da `hour == 0` no NB02 produz uma série
agregada iniciando em `bucket_id = 0`, sem ruptura artificial do início da linha do tempo.

## 12. Encaminhamento do Pipeline

Com a base agregada e a série contínua devidamente persistidas, o pipeline segue para:
- NB04 — Detecção de episódios críticos;
- NB05 — Engenharia de atributos;
- etapas posteriores de rotulagem e modelagem supervisionada.

Nesta execução, os artefatos salvos foram:
- `window_5min_base.parquet`
- `window_5min_series.parquet`
- `window_5min_base_keep_hour0.parquet`
- `window_5min_series_keep_hour0.parquet`

Esses arquivos passam a constituir a referência oficial da etapa de agregação temporal
para o cenário `keep_hour0`.